In [26]:
import json
import os
import re
import csv
import base64
import tempfile
from pathlib import Path
from typing import List, Optional
from dotenv import load_dotenv
load_dotenv('../.env', override=True)

from pypdf import PdfReader, PdfWriter
from pydantic import BaseModel, Field
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage

GCP_PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT", "project-6292083d-1bcd-4e65-bf9")
LOCATION    = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
MODEL_NAME  = "gemini/gemini-3-flash-preview"

llm = ChatVertexAI(
    model=MODEL_NAME,
    project=GCP_PROJECT,
    location=LOCATION,
    temperature=0.1,
)
print(f"Using model: {MODEL_NAME} | project: {GCP_PROJECT} | location: {LOCATION}")


Using model: gemini/gemini-3-flash-preview | project: project-6292083d-1bcd-4e65-bf9 | location: us-central1


/var/folders/s4/dzqp3h_n4k17cqyfz623c1480000gn/T/ipykernel_79130/3252936932.py:21: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(


In [27]:
DICTFILE = '../inputs/carolinian_full.pdf'
PAGES = [494, 1229]  # Change to [494, 1229] for full run


In [28]:
class TextItem(BaseModel):
    text: str = Field(..., description="The text content, without any dialect marker.")
    style: str = Field(..., description="'bold' if this is an English concept heading, 'italic' if this is a Carolinian headword.")
    dialect: str = Field("", description="Dialect marker if present in parentheses immediately after the italic word, e.g. TAN, LN, EL, S. Empty string if none.")


class Page(BaseModel):
    items: list[TextItem]
    number: int
    file: str

    @classmethod
    def load_pages(cls, filepath: str = "finderlist_pages.jsonl") -> list["Page"]:
        with open(filepath, "r") as f:
            return [cls.model_validate_json(line) for line in f]

    @staticmethod
    def save_pages(pages: list["Page"], filepath: str = "finderlist_pages.jsonl") -> None:
        with open(filepath, "w") as f:
            for page in pages:
                f.write(page.model_dump_json() + "\n")


In [29]:
def is_already_extracted(page_number: int, pages: List[Page]) -> bool:
    return any(page.number == page_number for page in pages)


In [30]:
def extract_entries(pdf_path: str, page_number: int) -> Page:
    reader = PdfReader(pdf_path)
    page = reader.pages[page_number]
    text = page.extract_text()
    writer = PdfWriter()
    writer.add_page(page)
    with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmpfile:
        writer.write(tmpfile)
        tmpfile_path = tmpfile.name

    pdf_bytes = Path(tmpfile_path).read_bytes()
    encoded_data = base64.b64encode(pdf_bytes).decode("utf-8")
    os.remove(tmpfile_path)

    prompt = (
        "<page_text>\n" + text + "\n</page_text>\n\n"
        "This is a page from the English-Carolinian finder list of a bilingual dictionary. "
        "Extract all text items in reading order. "
        "Bold words are English concept headings (style='bold'). "
        "Italic words are Carolinian headwords (style='italic'). "
        "After each italic Carolinian headword, there may be a dialect marker in parentheses "
        "in small caps (TAN, LN, EL, S) — capture this in the dialect field. "
        "All other normal (non-bold, non-italic) text is English description — do NOT include it. "
        "Return only bold and italic items as a JSON array with keys: text, style, dialect."
    )

    message = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "media", "mime_type": "application/pdf",
         "data": encoded_data},
    ])

    response = llm.invoke([message])
    content = response.content

    # Strip markdown fences if present
    content = re.sub(r"^```(?:json)?\n?", "", content.strip())
    content = re.sub(r"\n?```$", "", content.strip())

    # Fix invalid unicode escapes
    content = re.sub(r"\\u(?![0-9a-fA-F]{4})", r"\\\\u", content)

    items_data = json.loads(content)
    if isinstance(items_data, dict):
        items_data = next(iter(items_data.values()))

    return Page(
        items=[TextItem(**e) for e in items_data],
        number=page_number,
        file=pdf_path
    )


In [31]:
try:
    pages = Page.load_pages()
    print(f"Loaded {len(pages)} already-extracted pages")
except FileNotFoundError:
    pages = []
    print("No existing pages found, starting fresh")


Loaded 726 already-extracted pages


In [32]:
# Pages to overwrite (0-indexed)
OVERWRITE_PAGES = [764, 888, 557, 894, 796, 889, 989, 631, 653]

pages = [p for p in pages if p.number not in OVERWRITE_PAGES]
Page.save_pages(pages)
print(f"Removed {len(OVERWRITE_PAGES)} pages from cache, will re-extract them.")


Removed 9 pages from cache, will re-extract them.


In [ ]:
from tqdm import tqdm
import time

for page_number in tqdm(range(*PAGES)):
    if is_already_extracted(page_number, pages):
        print(f"Page {page_number} already extracted, skipping.")
        continue

    try:
        page = extract_entries(DICTFILE, page_number)
    except json.JSONDecodeError as e:
        print(f"JSON parse error on page {page_number}: {e}, skipping.")
        continue
    except Exception as e:
        print(f"Error on page {page_number}: {e}, waiting 60s and retrying...")
        time.sleep(60)
        try:
            page = extract_entries(DICTFILE, page_number)
        except Exception as e2:
            print(f"Failed again on page {page_number}: {e2}, skipping.")
            continue

    print(f"Page {page_number}: {len(page.items)} items")
    pages.append(page)
    Page.save_pages(pages)

print("Done.")


  0%|                                                   | 0/735 [00:00<?, ?it/s]/Users/temuulenkh/Documents/github/MUDIDI/.venv/lib/python3.14/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Page 494 already extracted, skipping.
Page 495 already extracted, skipping.
Page 496 already extracted, skipping.
Page 497 already extracted, skipping.
Page 498 already extracted, skipping.
Page 499 already extracted, skipping.
Page 500 already extracted, skipping.
Page 501 already extracted, skipping.
Page 502 already extracted, skipping.
Page 503 already extracted, skipping.
Page 504 already extracted, skipping.
Page 505 already extracted, skipping.
Page 506 already extracted, skipping.
Page 507 already extracted, skipping.
Page 508 already extracted, skipping.
Page 509 already extracted, skipping.
Page 510 already extracted, skipping.
Page 511 already extracted, skipping.
Page 512 already extracted, skipping.
Page 513 already extracted, skipping.
Page 514 already extracted, skipping.
Page 515 already extracted, skipping.
Page 516 already extracted, skipping.
Page 517 already extracted, skipping.
Page 518 already extracted, skipping.
Page 519 already extracted, skipping.
Page 520 alr

Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it raised InvalidArgument: 400 Invalid resource field value in the request. [reason: "RESOURCE_PROJECT_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "aiplatform.googleapis.com"
}
metadata {
  key: "method"
  value: "google.cloud.aiplatform.v1beta1.PredictionService.GenerateContent"
}
].
Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it raised InvalidArgument: 400 Invalid resource field value in the request. [reason: "RESOURCE_PROJECT_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "aiplatform.googleapis.com"
}
metadata {
  key: "method"
  value: "google.cloud.aiplatform.v1beta1.PredictionService.GenerateContent"
}
].
Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it

Error on page 557: 400 Invalid resource field value in the request. [reason: "RESOURCE_PROJECT_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "aiplatform.googleapis.com"
}
metadata {
  key: "method"
  value: "google.cloud.aiplatform.v1beta1.PredictionService.GenerateContent"
}
], waiting 60s and retrying...


Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it raised InvalidArgument: 400 Invalid resource field value in the request. [reason: "RESOURCE_PROJECT_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "aiplatform.googleapis.com"
}
metadata {
  key: "method"
  value: "google.cloud.aiplatform.v1beta1.PredictionService.GenerateContent"
}
].
Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it raised InvalidArgument: 400 Invalid resource field value in the request. [reason: "RESOURCE_PROJECT_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "aiplatform.googleapis.com"
}
metadata {
  key: "method"
  value: "google.cloud.aiplatform.v1beta1.PredictionService.GenerateContent"
}
].
Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it

...

Error on page 894: 400 Invalid resource field value in the request. [reason: "RESOURCE_PROJECT_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "aiplatform.googleapis.com"
}
metadata {
  key: "method"
  value: "google.cloud.aiplatform.v1beta1.PredictionService.GenerateContent"
}
], waiting 60s and retrying...


Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it raised InvalidArgument: 400 Invalid resource field value in the request. [reason: "RESOURCE_PROJECT_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "aiplatform.googleapis.com"
}
metadata {
  key: "method"
  value: "google.cloud.aiplatform.v1beta1.PredictionService.GenerateContent"
}
].
Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it raised InvalidArgument: 400 Invalid resource field value in the request. [reason: "RESOURCE_PROJECT_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "aiplatform.googleapis.com"
}
metadata {
  key: "method"
  value: "google.cloud.aiplatform.v1beta1.PredictionService.GenerateContent"
}
].
Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it

Failed again on page 894: 400 Invalid resource field value in the request. [reason: "RESOURCE_PROJECT_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "aiplatform.googleapis.com"
}
metadata {
  key: "method"
  value: "google.cloud.aiplatform.v1beta1.PredictionService.GenerateContent"
}
], skipping.
Page 895 already extracted, skipping.
Page 896 already extracted, skipping.
Page 897 already extracted, skipping.
Page 898 already extracted, skipping.
Page 899 already extracted, skipping.
Page 900 already extracted, skipping.
Page 901 already extracted, skipping.
Page 902 already extracted, skipping.
Page 903 already extracted, skipping.
Page 904 already extracted, skipping.
Page 905 already extracted, skipping.
Page 906 already extracted, skipping.
Page 907 already extracted, skipping.
Page 908 already extracted, skipping.
Page 909 already extracted, skipping.
Page 910 already extracted, skipping.
Page 911 already extracted, skipping.
Page 912 already extracted, skip

Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it raised InvalidArgument: 400 Invalid resource field value in the request. [reason: "RESOURCE_PROJECT_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "aiplatform.googleapis.com"
}
metadata {
  key: "method"
  value: "google.cloud.aiplatform.v1beta1.PredictionService.GenerateContent"
}
].
Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it raised InvalidArgument: 400 Invalid resource field value in the request. [reason: "RESOURCE_PROJECT_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "aiplatform.googleapis.com"
}
metadata {
  key: "method"
  value: "google.cloud.aiplatform.v1beta1.PredictionService.GenerateContent"
}
].
Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it

Error on page 989: 400 Invalid resource field value in the request. [reason: "RESOURCE_PROJECT_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "aiplatform.googleapis.com"
}
metadata {
  key: "method"
  value: "google.cloud.aiplatform.v1beta1.PredictionService.GenerateContent"
}
], waiting 60s and retrying...


## Post-processing: group italic Carolinian headwords under bold English concepts

In [ ]:
pages_sorted = sorted(pages, key=lambda p: p.number)

entries = []
current_english = None

for page in pages_sorted:
    for item in page.items:
        if item.style == "bold":
            current_english = item.text.strip()
        elif item.style == "italic" and current_english:
            entries.append({
                "english_word": current_english,
                "carolinian_headword": item.text.strip(),
                "dialect": item.dialect.strip(),
                "page_number": page.number,
            })

print(f"Total entries: {len(entries)}")
for e in entries[:5]:
    print(e)


In [ ]:
with open("finderlist.tsv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["english_word", "carolinian_headword", "dialect", "page_number"], delimiter="\t")
    writer.writeheader()
    writer.writerows(entries)

print(f"Saved {len(entries)} entries → finderlist.tsv")
